# 01 — Preparación de datos

Filtramos el DENUE para quedarnos solo con **escuelas en la Ciudad de México**
de los niveles de interés (preescolar, primaria, secundaria general, media
superior y media técnica terminal — sector público y privado).

Reproyectamos a UTM 14N (EPSG:32614) para tener coordenadas en metros.


In [1]:
import unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
from pyproj import Transformer

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CSV = ROOT / 'denue_inegi_61_.csv'
OUT = ROOT / 'data' / 'processed' / 'escuelas_cdmx.parquet'
OUT.parent.mkdir(parents=True, exist_ok=True)
print('csv:', CSV.exists(), CSV)


csv: True c:\Users\eduar\Documents\Universidad\Proyecto Hugo\denue_inegi_61_.csv


In [2]:
COLS = ['id','nom_estab','codigo_act','nombre_act','per_ocu','cve_ent','entidad',
        'cve_mun','municipio','latitud','longitud','fecha_alta']
df = pd.read_csv(CSV, encoding='latin-1', low_memory=False, usecols=COLS)
print('filas totales:', len(df))
df.head(2)


filas totales: 148406


,id,nom_estab,codigo_act,nombre_act,per_ocu,cve_ent,entidad,cve_mun,municipio,latitud,longitud,fecha_alta
0,46594,2010 BICENTENARIO DE LA INDEPENDENCIA DE MEXIC...,611122,Escuelas de educación primaria del sector público,11 a 30 personas,1,Aguascalientes,1,Aguascalientes,21.931754,-102.255813,2014-12
1,10712086,5678 ESTUDIO DE DANZA,611611,Escuelas de arte del sector privado,6 a 10 personas,1,Aguascalientes,1,Aguascalientes,21.870707,-102.311026,2024-11


In [3]:
# Filtro CDMX
df_cdmx = df[df['entidad'].str.contains('iudad de M', na=False)].copy()
print('CDMX:', len(df_cdmx))


CDMX: 10448


In [4]:
# Definición de las 12 categorías objetivo (claves SCIAN)
# 611111 preescolar publ, 611112 preescolar priv
# 611121 primaria publ,   611122 primaria priv  (cuidado: revisar abajo)
# Mejor usar el texto exacto del DENUE (con codificación latin-1 ya decodificada).
target_acts = [
    'Escuelas de educación preescolar del sector público',
    'Escuelas de educación preescolar del sector privado',
    'Escuelas de educación primaria del sector público',
    'Escuelas de educación primaria del sector privado',
    'Escuelas de educación secundaria general del sector público',
    'Escuelas de educación secundaria general del sector privado',
    'Escuelas de educación media superior del sector público',
    'Escuelas de educación media superior del sector privado',
    'Escuelas de educación media técnica terminal del sector público',
    'Escuelas de educación media técnica terminal del sector privado',
]

df_esc = df_cdmx[df_cdmx['nombre_act'].isin(target_acts)].copy()
print('escuelas objetivo en CDMX:', len(df_esc))
print(df_esc['nombre_act'].value_counts())


escuelas objetivo en CDMX: 5587
nombre_act
Escuelas de educación primaria del sector público                  1927
Escuelas de educación preescolar del sector público                1262
Escuelas de educación preescolar del sector privado                 992
Escuelas de educación secundaria general del sector público         613
Escuelas de educación primaria del sector privado                   348
Escuelas de educación media superior del sector público             204
Escuelas de educación media superior del sector privado             121
Escuelas de educación secundaria general del sector privado          84
Escuelas de educación media técnica terminal del sector privado      33
Escuelas de educación media técnica terminal del sector público       3
Name: count, dtype: int64


In [5]:
# Validar lat/lon dentro del bounding box de CDMX
mask_geo = (
    df_esc['latitud'].between(19.0, 19.7) &
    df_esc['longitud'].between(-99.4, -98.9)
)
print('descartadas por lat/lon fuera de CDMX:', (~mask_geo).sum())
df_esc = df_esc[mask_geo].copy()


descartadas por lat/lon fuera de CDMX: 0


In [6]:
# Derivar columnas 'nivel' y 'sector'
def parse_nivel(s: str) -> str:
    s = s.lower()
    if 'preescolar' in s: return 'preescolar'
    if 'primaria' in s: return 'primaria'
    if 'secundaria' in s: return 'secundaria'
    if 'media superior' in s: return 'media_superior'
    if 'media técnica' in s or 'media tecnica' in s: return 'media_tecnica'
    return 'otro'

def parse_sector(s: str) -> str:
    return 'privado' if 'privado' in s.lower() else 'público'

df_esc['nivel'] = df_esc['nombre_act'].map(parse_nivel)
df_esc['sector'] = df_esc['nombre_act'].map(parse_sector)
print(df_esc.groupby(['nivel','sector']).size().unstack(fill_value=0))


sector          privado  público
nivel                           
media_superior      121      204
media_tecnica        33        3
preescolar          992     1262
primaria            348     1927
secundaria           84      613


In [7]:
# Reproyección a UTM 14N (EPSG:32614) — coordenadas en metros
tr = Transformer.from_crs(4326, 32614, always_xy=True)
x, y = tr.transform(df_esc['longitud'].values, df_esc['latitud'].values)
df_esc['x_utm'] = x
df_esc['y_utm'] = y
df_esc[['latitud','longitud','x_utm','y_utm']].describe()


,latitud,longitud,x_utm,y_utm
count,5587.000000,5587.000000,5587.000000,5.587000e+03
mean,19.379175,-99.128246,486533.542070,2.142790e+06
std,0.080384,0.070751,7430.169475,8.894855e+03
min,19.133882,-99.334551,464853.552762,2.115651e+06
25%,19.325814,-99.177285,481381.424220,2.136886e+06
50%,19.374199,-99.128950,486455.736942,2.142240e+06
75%,19.442990,-99.075464,492078.130460,2.149851e+06
max,19.578135,-98.951644,505082.529828,2.164804e+06


In [8]:
# Guardar
df_esc.reset_index(drop=True).to_parquet(OUT, index=False)
print('guardado:', OUT, '|', len(df_esc), 'filas')


guardado: c:\Users\eduar\Documents\Universidad\Proyecto Hugo\data\processed\escuelas_cdmx.parquet | 5587 filas


## Verificación

- Esperamos ~5000–6000 escuelas tras el filtro.
- Cada nivel debe estar presente; preescolar y primaria dominan.
- Las coordenadas UTM x deben estar en el rango ~470000–510000, y en ~2120000–2170000.
